# Medical-Chatbot Colab Evaluation (GenerationService)

Pipeline:
1. Load data/raw/VL_..._...json (VL merged dataset file)
2. Split by `q_type` (1, 2, 3)
3. Mode A (`LLM only`) generation with Qwen2.5-Instruct
4. Mode B (`LLM + RAG`) generation with Qwen2.5-Instruct + `chromadb/`
5. Evaluate with `src/evaluation/metrics.py`


In [ ]:
# ===== 0) clone repository (first run only) =====
!git clone -b hahyun https://github.com/ljhljh0703-cmd/Medical-Chatbot.git /content/Medical-Chatbot
%cd /content/Medical-Chatbot

In [ ]:
# ===== 1) Install dependencies (first run only) =====
!pip -q install -U pip
!pip -q install -r requirements.txt

In [ ]:
# ===== 2) Paths and runtime settings =====
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_PATH = REPO_ROOT / 'data/raw/VL_내과_통합.json'
CHROMA_PATH = REPO_ROOT / 'chroma_db'
OUTPUT_DIR = REPO_ROOT / 'outputs/colab_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f'Missing data file: {DATA_PATH}'
assert CHROMA_PATH.exists(), f'Missing ChromaDB path: {CHROMA_PATH}'
print('REPO_ROOT =', REPO_ROOT)
print('DATA_PATH =', DATA_PATH)
print('CHROMA_PATH =', CHROMA_PATH)

In [ ]:
# ===== 3) Load data and split by q_type =====
import json
import pandas as pd

rows = json.loads(DATA_PATH.read_text(encoding='utf-8'))
df = pd.DataFrame(rows)
df['q_type'] = df['q_type'].astype(int)

required_cols = ['qa_id', 'q_type', 'question', 'answer']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')

split_by_qtype = {q: g.reset_index(drop=True) for q, g in df.groupby('q_type')}
for q in [1, 2, 3]:
    print(f'q_type={q}:', len(split_by_qtype.get(q, pd.DataFrame())))

df.head(2)

In [ ]:
# ===== 4) Initialize GenerationService and RAG/LoRA helpers =====
from config.settings import settings
from llm.generation_service import GenerationService
from retrieval.dense.embedder import Embedder
from retrieval.dense.chroma_store import ChromaStore
from retrieval.dense.dense_retriever import DenseRetriever
from retrieval.sparse.bm25_retriever import BM25Retriever
from retrieval.hybrid.fusion import HybridFusion

settings.model_backend = 'qwen'
settings.model_path = 'Qwen/Qwen2.5-7B-Instruct'
settings.model_mode = 'A'
settings.max_new_tokens = 256
settings.temperature = 0.2

# Retrieval embedding model.
# If OPENAI_API_KEY is missing, Embedder falls back to local sentence-transformers model.
settings.embedding_model = os.getenv('EMBEDDING_MODEL', 'jhgan/ko-sroberta-multitask')
settings.openai_api_key = os.getenv('OPENAI_API_KEY', settings.openai_api_key)

raw_lora_adapter_path = os.getenv('LORA_ADAPTER_PATH', 'src/training/lora_final')
lora_adapter_path = Path(raw_lora_adapter_path)
if not lora_adapter_path.is_absolute():
    lora_adapter_path = (REPO_ROOT / lora_adapter_path).resolve()
settings.lora_adapter_path = str(lora_adapter_path)

gen_service = GenerationService()
embedder = Embedder(embedding_model=settings.embedding_model, api_key=settings.openai_api_key)
store = ChromaStore(collection_name=settings.chroma_collection, persist_directory=str(CHROMA_PATH))
dense_retriever = DenseRetriever(embedder=embedder, store=store)
bm25_retriever = BM25Retriever(collection_name=settings.chroma_collection, db_path=str(CHROMA_PATH))
hybrid_retriever = HybridFusion(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
    alpha=0.5,
    fusion_method='weighted_sum',
)

print('model_path =', settings.model_path)
print('embedding_model =', settings.embedding_model)
print('chroma_collection =', settings.chroma_collection)
print('chroma_count =', store.count())
print('lora_adapter_path =', settings.lora_adapter_path)




In [ ]:
# ===== 5) Generation functions for Mode A / Mode B1 / Mode B2 / Mode B3 / Mode C =====
from typing import Optional
import time
from tqdm.auto import tqdm
import llm.generation_service as generation_service_module

QTYPE_SYSTEM_PROMPTS = {
    1: """당신은 한국 의학 국가고시 스타일 문제를 푸는 내과 전문의입니다.

[문항 유형]
- 객관식 5지선다 (q_type=1)

[답변 규칙]
- 반드시 보기 번호 1)~5) 중 하나만 선택해 답변합니다.
- 출력은 정답 한 줄만 작성합니다. 예: "3) 급성 심낭염"
- 이유, 해설, 인사말, 불필요한 부연 설명은 작성하지 않습니다.
- 문제에 제시된 선택지 밖의 답을 새로 만들지 않습니다.

[판단 기준]
- 증상, 병력, 활력징후, 검사 소견 등 임상 단서를 우선 반영해 가장 타당한 보기 하나를 고릅니다.
""",

    2: """당신은 내과 단답형 문항에 답하는 의학 어시스턴트입니다.

[문항 유형]
- 단답형 (q_type=2)

[답변 규칙]
- 정답 핵심어만 매우 간결하게 답합니다.
- 기본적으로 명사형(질환명, 약물명, 검사명, 해부학 용어)으로 작성합니다.
- 문장형 설명, 근거, 부연 설명은 작성하지 않습니다.
- 가능하면 1~5단어 이내로 답합니다.
- 필요할 때만 최소한의 괄호를 사용합니다.

[출력 예시 형식]
- "ACE 억제제"
- "아스피린"
- "나트륨"
""",

    3: """당신은 내과 서술형 문항에 답하는 전문의입니다.

[문항 유형]
- 서술형 (q_type=3)

[답변 규칙]
- 답변은 반드시 한국어만 사용합니다.
- 중국어(간체/번체), 일본어, 영어 문장으로 답하지 않습니다.
- 질문에서 요구한 항목을 빠짐없이 구조적으로 서술합니다.
- 임상적으로 중요한 기준(진단 기준, 중증도 기준, 치료 원칙/순서)을 명확히 포함합니다.
- 불필요하게 장황하지 않게, 핵심 위주로 문단 또는 번호를 사용해 정리합니다.
- 근거 없는 추측은 피하고, 제시된 정보와 일반적인 내과 원칙에 기반해 답합니다.
"""
}


def get_system_prompt_for_qtype(q_type: int) -> str:
    return QTYPE_SYSTEM_PROMPTS.get(int(q_type), QTYPE_SYSTEM_PROMPTS[3])


def _to_query_vec(question: str):
    vec = embedder.embed(question)
    if isinstance(vec, list) and len(vec) > 0 and isinstance(vec[0], list):
        return vec[0]
    return vec


def build_rag_context_dense(question: str, top_k: int = 5) -> str:
    q_vec = _to_query_vec(question)
    results = store.query(query_embedding=q_vec, top_k=top_k)

    ids = (results.get('ids') or [[]])[0]
    docs = (results.get('documents') or [[]])[0]
    metas = (results.get('metadatas') or [[]])[0]
    dists = (results.get('distances') or [[]])[0]

    if not ids:
        return ''

    parts = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
        meta = meta or {}
        source = meta.get('source_spec', 'unknown')
        sim = max(0.0, 1.0 - (dist / 2.0))
        parts.append(f'[Ref {i}] (source: {source}, score: {sim:.3f})\n{doc}')
    return '\n\n'.join(parts)


def build_rag_context_bm25(question: str, top_k: int = 5) -> str:
    results = bm25_retriever.retrieve(question, top_k=top_k)
    if not results:
        return ''

    max_score = max((r.get('bm25_score', 0.0) for r in results), default=1.0) or 1.0
    parts = []
    for i, r in enumerate(results, 1):
        meta = r.get('metadata') or {}
        source = meta.get('source_spec', 'unknown')
        score = r.get('bm25_score', 0.0) / max_score
        parts.append(f'[Ref {i}] (source: {source}, score: {score:.3f})\n{r.get("text", "")}')
    return '\n\n'.join(parts)


def build_rag_context_hybrid(question: str, top_k: int = 5) -> str:
    chunks = hybrid_retriever.retrieve(question, top_k=top_k)
    if not chunks:
        return ''

    parts = []
    for i, chunk in enumerate(chunks, 1):
        source = chunk.source_spec or 'unknown'
        parts.append(
            f'[Ref {i}] (source: {source}, score: {chunk.similarity_score:.3f})\n{chunk.content_snippet}'
        )
    return '\n\n'.join(parts)


def build_rag_context(question: str, mode: str, top_k: int = 5) -> str:
    if mode == 'B1':
        return build_rag_context_dense(question, top_k=top_k)
    if mode == 'B2':
        return build_rag_context_bm25(question, top_k=top_k)
    if mode == 'B3':
        return build_rag_context_hybrid(question, top_k=top_k)
    return ''


def get_generation_mode(mode: str) -> str:
    if mode == 'A':
        return 'A'
    if mode == 'C':
        return 'C'
    return 'B'


def generate_answer(question: str, q_type: int, mode: str = 'A', top_k: int = 5) -> str:
    system_prompt = get_system_prompt_for_qtype(q_type)
    original_prompt = generation_service_module.SYSTEM_PROMPT
    generation_service_module.SYSTEM_PROMPT = system_prompt
    try:
        if mode == 'A':
            return gen_service.generate(query=question, context=None, mode='A')
        if mode == 'C':
            return gen_service.generate(query=question, context=None, mode='C')
        if mode in ('B1', 'B2', 'B3'):
            ctx = build_rag_context(question, mode=mode, top_k=top_k)
            return gen_service.generate(query=question, context=ctx, mode='B')
        raise ValueError('mode must be one of A, B1, B2, B3, C')
    finally:
        generation_service_module.SYSTEM_PROMPT = original_prompt


def run_generation(
    df_in,
    mode: str,
    q_type: int,
    top_k: int = 5,
    sample_n: Optional[int] = None,
    batch_size: int = 8,
):
    if batch_size <= 0:
        raise ValueError('batch_size must be >= 1')

    data = df_in.copy()
    if sample_n is not None:
        data = data.head(sample_n).copy()

    questions = data['question'].tolist()
    preds = []

    system_prompt = get_system_prompt_for_qtype(q_type)
    original_prompt = generation_service_module.SYSTEM_PROMPT
    generation_service_module.SYSTEM_PROMPT = system_prompt

    start_ts = time.perf_counter()

    try:
        for i in tqdm(range(0, len(questions), batch_size), desc=f'Generate mode={mode}, q_type={q_type}'):
            batch_questions = questions[i : i + batch_size]

            if mode in ('A', 'C'):
                batch_contexts = None
            elif mode in ('B1', 'B2', 'B3'):
                batch_contexts = [build_rag_context(q, mode=mode, top_k=top_k) for q in batch_questions]
            else:
                raise ValueError('mode must be one of A, B1, B2, B3, C')

            try:
                batch_preds = gen_service.generate(
                    query=batch_questions,
                    context=batch_contexts,
                    mode=get_generation_mode(mode),
                )

                if isinstance(batch_preds, str):
                    batch_preds = [batch_preds]
                else:
                    batch_preds = list(batch_preds)

                if len(batch_preds) != len(batch_questions):
                    raise ValueError(
                        f'Batch output size mismatch: got {len(batch_preds)}, expected {len(batch_questions)}'
                    )

            except Exception:
                batch_preds = []
                for idx, q in enumerate(batch_questions):
                    try:
                        ctx = None if batch_contexts is None else batch_contexts[idx]
                        batch_preds.append(gen_service.generate(query=q, context=ctx, mode=get_generation_mode(mode)))
                    except Exception as e:
                        batch_preds.append(f'[ERROR] {e}')

            preds.extend(batch_preds)
    finally:
        generation_service_module.SYSTEM_PROMPT = original_prompt

    elapsed_sec = time.perf_counter() - start_ts

    out = data.copy()
    out[f'pred_{mode}'] = preds
    return out, elapsed_sec, len(out)





In [ ]:
# ===== 6) Run Mode A / B1 / B2 / B3 / C =====
# For quick checks in Colab, start with a small sample.
SAMPLE_N_PER_QTYPE = 20  # set None for full run
TOP_K = 5

mode_names = ['A', 'B1', 'B2', 'B3', 'C']
mode_parts = {m: [] for m in mode_names}
runtime_records = []

for q in [1, 2, 3]:
    part = split_by_qtype.get(q)
    if part is None or len(part) == 0:
        continue

    print(f'q_type={q} prompt loaded')

    for mode in mode_names:
        out, elapsed_sec, n_items = run_generation(part, mode=mode, q_type=q, top_k=TOP_K, sample_n=SAMPLE_N_PER_QTYPE)
        runtime_records.append({
            'mode': mode,
            'q_type': q,
            'n': int(n_items),
            'total_runtime_sec': float(elapsed_sec),
        })
        keep_cols = ['qa_id', 'q_type', 'question', 'answer', f'pred_{mode}']
        mode_parts[mode].append(out[keep_cols])

base_mode = next((m for m in mode_names if mode_parts[m]), None)
pred_df = (
    pd.concat(mode_parts[base_mode], ignore_index=True).copy()
    if base_mode is not None
    else pd.DataFrame(columns=['qa_id', 'q_type', 'question', 'answer'])
)

for mode in mode_names:
    if mode == base_mode:
        continue
    pred_mode = pd.concat(mode_parts[mode], ignore_index=True) if mode_parts[mode] else pd.DataFrame()
    if not pred_mode.empty:
        pred_df = pred_df.merge(
            pred_mode[['qa_id', 'q_type', f'pred_{mode}']],
            on=['qa_id', 'q_type'],
            how='outer',
        )

runtime_df = pd.DataFrame(runtime_records)
if runtime_df.empty:
    runtime_summary_df = pd.DataFrame(columns=['mode', 'q_type', 'n', 'total_runtime_sec', 'avg_runtime_sec_per_item'])
else:
    runtime_by_q = (
        runtime_df.groupby(['mode', 'q_type'], as_index=False)
        .agg({'n': 'sum', 'total_runtime_sec': 'sum'})
    )
    runtime_by_q['avg_runtime_sec_per_item'] = runtime_by_q['total_runtime_sec'] / runtime_by_q['n'].clip(lower=1)

    runtime_all = (
        runtime_df.groupby(['mode'], as_index=False)
        .agg({'n': 'sum', 'total_runtime_sec': 'sum'})
    )
    runtime_all['q_type'] = 'all'
    runtime_all['avg_runtime_sec_per_item'] = runtime_all['total_runtime_sec'] / runtime_all['n'].clip(lower=1)

    runtime_summary_df = pd.concat([runtime_by_q, runtime_all], ignore_index=True)

runtime_summary_df.to_csv(OUTPUT_DIR / 'runtime_A_B1_B2_B3_C.csv', index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_DIR / 'runtime_A_B1_B2_B3_C.csv')

pred_df.to_csv(OUTPUT_DIR / 'predictions_A_B1_B2_B3_C.csv', index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_DIR / 'predictions_A_B1_B2_B3_C.csv')
pred_df.head(3)




In [ ]:
# ===== 7) Evaluate with metrics.py =====
from evaluation.metrics import exact_match, rouge_l, bert_score

def evaluate_with_metrics(df_eval, pred_col: str):
    refs = df_eval['answer'].fillna('').tolist()
    preds = df_eval[pred_col].fillna('').tolist()

    ems = [exact_match(p, r, q) for p, r, q in zip(preds, refs, df_eval['q_type'].tolist())]
    rls = [rouge_l(p, r) for p, r in zip(preds, refs)]
    bss = bert_score(preds, refs)

    scored = df_eval.copy()
    scored['exact_match'] = ems
    scored['rouge_l'] = rls
    scored['bert_score'] = bss

    summary_rows = []
    for q in [1, 2, 3]:
        sub = scored[scored['q_type'] == q]
        if len(sub) == 0:
            continue
        summary_rows.append({
            'mode': pred_col.replace('pred_', ''),
            'q_type': q,
            'n': len(sub),
            'exact_match': float(sub['exact_match'].mean()),
            'rouge_l': float(sub['rouge_l'].mean()),
            'bert_score': float(sub['bert_score'].mean()),
        })

    summary_rows.append({
        'mode': pred_col.replace('pred_', ''),
        'q_type': 'all',
        'n': len(scored),
        'exact_match': float(scored['exact_match'].mean()),
        'rouge_l': float(scored['rouge_l'].mean()),
        'bert_score': float(scored['bert_score'].mean()),
    })

    return scored, pd.DataFrame(summary_rows)

mode_names = ['A', 'B1', 'B2', 'B3', 'C']
summary_parts = []

for mode in mode_names:
    pred_col = f'pred_{mode}'
    if pred_col not in pred_df.columns:
        continue
    scored_mode, summary_mode = evaluate_with_metrics(
        pred_df[['qa_id', 'q_type', 'question', 'answer', pred_col]].copy(),
        pred_col,
    )
    summary_parts.append(summary_mode)
    scored_mode.to_csv(OUTPUT_DIR / f'scored_mode_{mode}.csv', index=False, encoding='utf-8-sig')
    print('saved:', OUTPUT_DIR / f'scored_mode_{mode}.csv')

summary_all = pd.concat(summary_parts, ignore_index=True)

if 'runtime_summary_df' in globals() and not runtime_summary_df.empty:
    summary_all = summary_all.merge(
        runtime_summary_df[['mode', 'q_type', 'total_runtime_sec', 'avg_runtime_sec_per_item']],
        on=['mode', 'q_type'],
        how='left',
    )

summary_all.to_csv(OUTPUT_DIR / 'summary_metrics_A_B1_B2_B3_C.csv', index=False, encoding='utf-8-sig')

print('saved:', OUTPUT_DIR / 'summary_metrics_A_B1_B2_B3_C.csv')
summary_all




In [ ]:
# ===== 8) Download output files =====
from pathlib import Path
import shutil

# Download all files generated under output_dir in Google Colab
output_dir = Path("outputs/colab_eval")
zip_base = output_dir.parent / output_dir.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(output_dir)))

from google.colab import files
files.download(str(zip_path))
print(f"Download ready: {zip_path}")

## Notes
- Set `SAMPLE_N_PER_QTYPE = None` for full evaluation.
- RAG quality depends on whether query embedding model matches the model used when building `chromadb/`.
- If Colab GPU memory is not enough, switch to a smaller Qwen model (for example, 3B).
- Mode C uses LoRA adapter path from `LORA_ADAPTER_PATH` (default: `src/training/lora_final`).

